In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# 设备配置（CPU/GPU）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

#### 几何参数

In [2]:
d0 = 0.05  # 内管直径（m）
d1 = 0.13  # 外壳直径（m）
r0 = d0 / 2  # 内管半径
r1 = d1 / 2  # 外壳半径

#### 材料热物性参数

In [3]:
# PCM（石蜡）
rho_s = 880.0    # 固相密度 (kg/m³)
rho_l = 760.0    # 液相密度 (kg/m³)
cp_s = 2180.0    # 固相定压比热容 (J/(kg·K))
cp_l = 2390.0    # 液相定压比热容 (J/(kg·K))
lambda_s = 0.4   # 固相导热系数 (W/(m·K))
lambda_l = 0.15  # 液相导热系数 (W/(m·K))
mu_l = 0.001     # 液相粘度 (kg/(m·s))
L = 255000.0     # 相变潜热 (J/kg)
Tpc = 316.15     # 相变温度 (K)
DeltaT = 6.0     # 相变温度区间 (K)

# 高导热材料（铜）
rho_Cu = 8960.0  # 密度 (kg/m³)
lambda_Cu = 400.0  # 导热系数 (W/(m·K))
cp_Cu = 385.0    # 定压比热容 (J/(kg·K))
mu_Cu = 1e10     # 铜为固体，粘度取极大值抑制流动

#### 物理模型参数

In [4]:
Am = 1e5         # 糊状区常数
epsilon = 0.001  # 避免分母为0
xi = 1.0         # 粘度函数常数
alpha = 1e-4     # PCM体胀系数 (1/K)
g = 9.81         # 重力加速度 (m/s²)

#### 拓扑优化参数

In [5]:
phi_total = 0.3  # 高导热材料体积比约束
case = 3         # 优化目标选择：1=平均温度，2=温度均方差，3=多目标
w1, w2, w3 = 1.0, 1.0, 1.0  # 多目标权重

L_char = d1 - d0  # 特征长度：环形域宽度 (m)
U_char = 1e-3     # 特征速度：1mm/s (m/s)
T_char = 70.0     # 特征温差：70K (K)
t_char = L_char / (U_char * np.sqrt(g * alpha * T_char * L_char))  # 特征时间 (s)
print(f"特征尺度: L_char={L_char:.4f}m, U_char={U_char:.4f}m/s, T_char={T_char:.1f}K, t_char={t_char:.1f}s")

特征尺度: L_char=0.0800m, U_char=0.0010m/s, T_char=70.0K, t_char=1079.3s


#### 训练超参数 

In [6]:
N_mass = 10000   # 质量守恒方程采样点数量
N_mom = 10000    # 动量方程采样点数量
N_heat = 10000   # 传热方程采样点数量
N_IC = 8000      # 初始条件采样点数量
N_BC1 = 3000     # 内管壁边界采样点数量
N_BC2 = 3000     # 外壳边界采样点数量
N_rho = 10000    # 拓扑设计变量采样点数量
N_vol = 5000     # 体积比约束采样点数量
N_obj = 10000    # 优化目标采样点数量

lambda1 = 1.0    # PDE损失权重
lambda2 = 1.0    # IC/BC损失权重
lambda3 = 100.0  # 拓扑约束损失权重
lambda4 = 0.1    # 优化目标损失权重

In [7]:
class ResidualBlock(nn.Module):
    """残差块：缓解深层网络梯度消失，提升表达能力"""
    def __init__(self, dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.tanh = nn.Tanh()
    
    def forward(self, x):
        residual = x  # 残差连接：保留输入特征
        out = self.tanh(self.fc1(x))
        out = self.fc2(out)
        return self.tanh(out + residual)  # 残差相加后激活

In [8]:
class TopoPINN(nn.Module):
    def __init__(self, hidden_layers=6, hidden_dim=256):
        super(TopoPINN, self).__init__()
        # 主网络：处理随时间变化的变量 (ux, uy, p, T)
        self.main_net = nn.Sequential(
            nn.Linear(3, hidden_dim),  # 输入(x_star, y_star, τ_star)
            nn.Tanh(),
            *[ResidualBlock(hidden_dim) for _ in range(hidden_layers)],
            nn.Linear(hidden_dim, 4)  # 输出u_x*, u_y*, p*, T*
        )
        
        # 拓扑网络：仅处理空间变量，输出不随时间变化
        # 添加可学习的温度参数用于控制Sigmoid的陡度
        self.temperature = nn.Parameter(torch.tensor(0.1))
        
        self.topo_net = nn.Sequential(
            nn.Linear(2, hidden_dim),  # 输入(x_star, y_star)
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),  # 输出ρ_x原始值
        )
        # 初始化拓扑网络，使其初始输出接近0.3
        self._init_weights()
    
    def _init_weights(self):
        """初始化拓扑网络权重，使初始ρ_x接近目标体积比"""
        for layer in self.topo_net:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, mean=0.0, std=0.1)
                nn.init.constant_(layer.bias, 0.0)
    
    def forward(self, x):
        # x: [batch_size, 3]，对应(x_star, y_star, τ_star)
        # 提取空间坐标（无量纲）
        x_space = x[:, 0:2]
        
        # 主网络输出（无量纲变量）
        main_out = self.main_net(x)
        
        # 转换为有量纲变量
        # 速度：u* = u/U_char → u = u* * U_char
        ux = main_out[:, 0:1] * U_char
        uy = main_out[:, 1:2] * U_char
        
        # 压力：p* = p/(ρ_l * U_char²) → p = p* * ρ_l * U_char²
        p = main_out[:, 2:3] * rho_l * U_char**2
        
        # 温度：T* = (T - T0)/T_char → T = T* * T_char + T0
        T0 = 290.0  # 参考温度
        T = main_out[:, 3:4] * T_char + T0
        
        # 拓扑网络输出
        rho_x_raw = self.topo_net(x_space)
        # 使用温度参数控制Sigmoid陡度，避免梯度消失
        rho_x = torch.sigmoid(rho_x_raw / self.temperature)
        
        return ux, uy, p, T, rho_x
    
    def compute_phi_from_T(self, T):
        """根据温度计算液相率φ(T)"""
        T_lower = Tpc - DeltaT/2
        T_upper = Tpc + DeltaT/2
        phi = (T - T_lower) / (T_upper - T_lower)
        phi = torch.clamp(phi, 0.0, 1.0)
        return phi
    
    def compute_topo_regularization(self, x_space):
        """计算拓扑正则化项：平滑性和二值性"""
        # 空间梯度正则化（平滑性）
        x_space_requires_grad = x_space.requires_grad_(True)
        rho_x_raw = self.topo_net(x_space_requires_grad)
        rho_x = torch.sigmoid(rho_x_raw / self.temperature)
        
        # 计算空间梯度
        grad_rho = torch.autograd.grad(rho_x, x_space_requires_grad, 
                                       grad_outputs=torch.ones_like(rho_x),
                                       create_graph=True, retain_graph=True)[0]
        grad_norm = torch.sum(grad_rho**2, dim=1, keepdim=True)
        smooth_loss = torch.mean(grad_norm)
        
        # 二值性正则化（鼓励ρ_x接近0或1）
        binary_loss = torch.mean(rho_x * (1 - rho_x))
        
        return 0.01 * smooth_loss + 0.1 * binary_loss

In [9]:
def compute_thermo_props(T, rho_x):
    """
    计算混合材料的热物性参数（PCM+铜）
    输入：T（温度）、rho_x（拓扑设计变量）
    输出：rho_total, lambda_total, mu_total, cp_total, a（热扩散率）
    """
    # 液相率φ计算
    phi = torch.clamp((T - (Tpc - DeltaT/2)) / DeltaT, 0.0, 1.0)
    
    # PCM密度
    rho_PCM = rho_s + (rho_l - rho_s) * phi
    
    # 混合密度
    rho_total = rho_x * rho_Cu + (1.0 - rho_x) * rho_PCM
    
    # 混合导热系数：调和平均
    lambda_PCM = lambda_s + (lambda_l - lambda_s) * phi
    lambda_reciprocal = rho_x/(lambda_Cu + 1e-10) + (1.0 - rho_x)/(lambda_PCM + 1e-10)
    lambda_total = 1.0 / (lambda_reciprocal + 1e-10)
    
    # 糊状区源项和粘度
    S_t = Am * ((1.0 - phi) ** 2) / (phi ** 2 + epsilon)
    mu_PCM = mu_l + S_t * 1.0
    mu_PCM = torch.clamp(mu_PCM, 1e-6, 1e6)
    mu_total = rho_x * mu_Cu + (1.0 - rho_x) * mu_PCM
    nu = mu_total / (rho_total + 1e-10)
    
    # 定压比热容
    sigma = DeltaT / 4.0
    D_T = torch.exp(-((T - Tpc) ** 2) / (sigma ** 2)) / (torch.sqrt(torch.tensor(np.pi)) * sigma)
    D_T = torch.clamp(D_T, 0.0, 1e3)
    cp_PCM = cp_s + phi * (cp_l - cp_s) + L * D_T
    
    # 混合比热容
    mass_Cu = rho_x * rho_Cu
    mass_PCM = (1.0 - rho_x) * rho_PCM
    mass_total = mass_Cu + mass_PCM + 1e-10
    cp_total = (mass_Cu * cp_Cu + mass_PCM * cp_PCM) / mass_total
    
    # 热扩散率
    a = lambda_total / (rho_total * cp_total + 1e-10)
    
    return rho_total, lambda_total, mu_total, nu, cp_total, a, S_t

In [10]:
def sample_collocation_points(N):
    """采样设计域内配点（无量纲坐标）"""
    # 1. 时间采样：τ ∈ [0, t_char]，归一化到[0,1]
    tau = np.random.uniform(0.0, t_char, size=(N, 1))
    tau_star = tau / t_char  # 无量纲时间
    
    # 2. 空间采样：环形域
    r = np.random.uniform(r0, r1, size=(N, 1))
    theta = np.random.uniform(0.0, 2*np.pi, size=(N, 1))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    
    # 无量纲空间坐标
    x_star = x / L_char
    y_star = y / L_char
    
    # 面积元权重
    weight = r / np.mean(r)
    
    points = np.hstack([x_star, y_star, tau_star])
    return (torch.tensor(points, dtype=torch.float32).to(device),
            torch.tensor(weight, dtype=torch.float32).to(device))

def sample_initial_condition(N):
    """采样初始条件点"""
    tau_star = np.zeros((N, 1))
    r = np.random.uniform(r0, r1, size=(N, 1))
    theta = np.random.uniform(0.0, 2*np.pi, size=(N, 1))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    x_star = x / L_char
    y_star = y / L_char
    weight = r / np.mean(r)
    
    points = np.hstack([x_star, y_star, tau_star])
    return (torch.tensor(points, dtype=torch.float32).to(device),
            torch.tensor(weight, dtype=torch.float32).to(device))

def sample_boundary(N1, N2):
    """采样边界点"""
    # 内管壁
    theta1 = np.random.uniform(0.0, 2*np.pi, size=(N1, 1))
    x1 = r0 * np.cos(theta1)
    y1 = r0 * np.sin(theta1)
    x1_star = x1 / L_char
    y1_star = y1 / L_char
    tau1_star = np.random.uniform(0.0, 1.0, size=(N1, 1))
    bc1_points = np.hstack([x1_star, y1_star, tau1_star])
    
    # 外壳
    theta2 = np.random.uniform(0.0, 2*np.pi, size=(N2, 1))
    x2 = r1 * np.cos(theta2)
    y2 = r1 * np.sin(theta2)
    x2_star = x2 / L_char
    y2_star = y2 / L_char
    tau2_star = np.random.uniform(0.0, 1.0, size=(N2, 1))
    bc2_points = np.hstack([x2_star, y2_star, tau2_star])
    
    return (torch.tensor(bc1_points, dtype=torch.float32).to(device),
            torch.tensor(bc2_points, dtype=torch.float32).to(device))

In [11]:
def compute_volume_constraint_loss(model, collocation_points, collocation_weights, hard_penalty=True):
    """计算体积约束损失"""
    # 从配点中提取空间坐标
    x_space = collocation_points[:, 0:2].requires_grad_(True)
    x_with_time = torch.cat([x_space, torch.zeros_like(x_space[:, 0:1])], dim=1)
    
    # 获取拓扑变量
    _, _, _, _, rho_x = model(x_with_time)
    
    # 计算体积比
    rho_x_avg = torch.sum(rho_x * collocation_weights) / torch.sum(collocation_weights)
    
    # 硬约束惩罚
    if hard_penalty:
        if rho_x_avg > phi_total + 0.01:  # 超过1%容忍度
            vol_residual = (rho_x_avg - phi_total) ** 2 * 1000.0
        else:
            vol_residual = (rho_x_avg - phi_total) ** 2 * 10.0
    else:
        vol_residual = (rho_x_avg - phi_total) ** 2 * 10.0
    
    return vol_residual, rho_x_avg

In [12]:
def compute_loss(model, collocation_points, collocation_weights, ic_points, ic_weights, 
                bc1_points, bc2_points, heat_storage=True, compute_objective=True,
                stage="pretrain"):
    """计算总损失"""
    # 预处理
    model.train()
    x_col = collocation_points.requires_grad_(True)
    ux_col, uy_col, p_col, T_col, rho_x_col = model(x_col)
    rho_x_col = torch.clamp(rho_x_col, 1e-6, 1.0 - 1e-6)
    T_col = torch.clamp(T_col, 285.0, 365.0)
    rho_total, lambda_total, mu_total, nu_col, cp_total, a_col, S_t = compute_thermo_props(T_col, rho_x_col)
    
    # 获取其他点输出
    x_ic = ic_points.requires_grad_(True)
    ux_ic, uy_ic, p_ic, T_ic, rho_x_ic = model(x_ic)
    x_bc1 = bc1_points.requires_grad_(True)
    ux_bc1, uy_bc1, p_bc1, T_bc1, rho_x_bc1 = model(x_bc1)
    x_bc2 = bc2_points.requires_grad_(True)
    ux_bc2, uy_bc2, p_bc2, T_bc2, rho_x_bc2 = model(x_bc2)
    
    # ==================== PDE损失 ====================
    # 质量守恒
    grad_ux = torch.autograd.grad(ux_col, x_col, grad_outputs=torch.ones_like(ux_col),
                                  create_graph=True, retain_graph=True)[0]
    dudx_norm = grad_ux[:, 0:1] / L_char  # 转换为实际导数
    dudy_norm = grad_ux[:, 1:2] / L_char
    grad_uy = torch.autograd.grad(uy_col, x_col, grad_outputs=torch.ones_like(uy_col),
                                  create_graph=True, retain_graph=True)[0]
    dvdx_norm = grad_uy[:, 0:1] / L_char
    dvdy_norm = grad_uy[:, 1:2] / L_char
    
    mass_residual = dudx_norm + dvdy_norm
    L_mass = torch.sum(mass_residual ** 2 * collocation_weights) / torch.sum(collocation_weights)
    
    # 动量守恒
    convect_ux = ux_col * dudx_norm + uy_col * dudy_norm
    convect_uy = ux_col * dvdx_norm + uy_col * dvdy_norm
    
    grad_p = torch.autograd.grad(p_col, x_col, grad_outputs=torch.ones_like(p_col),
                                 create_graph=True, retain_graph=True)[0]
    dpdx_norm = grad_p[:, 0:1] / L_char
    dpdy_norm = grad_p[:, 1:2] / L_char
    pressure_term_x = -dpdx_norm / rho_total
    pressure_term_y = -dpdy_norm / rho_total
    
    # 二阶导数计算
    d2udx2_norm = torch.autograd.grad(dudx_norm, x_col, grad_outputs=torch.ones_like(dudx_norm),
                                       create_graph=True, retain_graph=True)[0][:, 0:1] / L_char
    d2udy2_norm = torch.autograd.grad(dudy_norm, x_col, grad_outputs=torch.ones_like(dudy_norm),
                                       create_graph=True, retain_graph=True)[0][:, 1:2] / L_char
    viscous_term_x = nu_col * (d2udx2_norm + d2udy2_norm)
    
    d2vdx2_norm = torch.autograd.grad(dvdx_norm, x_col, grad_outputs=torch.ones_like(dvdx_norm),
                                       create_graph=True, retain_graph=True)[0][:, 0:1] / L_char
    d2vdy2_norm = torch.autograd.grad(dvdy_norm, x_col, grad_outputs=torch.ones_like(dvdy_norm),
                                       create_graph=True, retain_graph=True)[0][:, 1:2] / L_char
    viscous_term_y = nu_col * (d2vdx2_norm + d2vdy2_norm)
    
    source_term_x = -S_t * ux_col
    source_term_y = -S_t * uy_col
    
    # 浮力项（修正）
    F_B = rho_l * alpha * g * (T_col - Tpc) * (1.0 - rho_x_col)
    
    mom_residual_x = convect_ux + pressure_term_x - viscous_term_x - source_term_x
    mom_residual_y = convect_uy + pressure_term_y - viscous_term_y - source_term_y - F_B
    L_momentum = torch.sum((mom_residual_x ** 2 + mom_residual_y ** 2) * collocation_weights) / torch.sum(collocation_weights)
    
    # 传热控制
    grad_T = torch.autograd.grad(T_col, x_col, grad_outputs=torch.ones_like(T_col),
                                 create_graph=True, retain_graph=True)[0]
    dTdx_norm = grad_T[:, 0:1] / L_char
    dTdy_norm = grad_T[:, 1:2] / L_char
    dTdtau_star = grad_T[:, 2:3]
    
    # 时间导数修正：∂T/∂τ = ∂T/∂τ* * dτ*/dτ = dTdtau_star / t_char
    dTdtau_real = dTdtau_star / t_char
    
    convect_T = ux_col * dTdx_norm + uy_col * dTdy_norm
    
    d2Tdx2_norm = torch.autograd.grad(dTdx_norm, x_col, grad_outputs=torch.ones_like(dTdx_norm),
                                       create_graph=True, retain_graph=True)[0][:, 0:1] / L_char
    d2Tdy2_norm = torch.autograd.grad(dTdy_norm, x_col, grad_outputs=torch.ones_like(dTdy_norm),
                                       create_graph=True, retain_graph=True)[0][:, 1:2] / L_char
    diffusive_term = a_col * (d2Tdx2_norm + d2Tdy2_norm)
    
    heat_residual = dTdtau_real + convect_T - diffusive_term
    L_heat = torch.sum(heat_residual ** 2 * collocation_weights) / torch.sum(collocation_weights)
    
    L_pressure_reg = torch.mean(p_col ** 2) * 1e-6
    L_PDE = L_mass + L_momentum + L_heat + L_pressure_reg
    
    # ==================== IC/BC损失 ====================
    T0 = 290.0 if heat_storage else 360.0
    L_IC = torch.sum(((T_ic - T0) ** 2 * 10 + ux_ic ** 2 + uy_ic ** 2) * ic_weights) / torch.sum(ic_weights)
    
    Tw = 360.0 if heat_storage else 290.0
    L_BC1 = torch.mean((T_bc1 - Tw) ** 2 * 2)
    
    grad_T_bc2 = torch.autograd.grad(T_bc2, x_bc2, grad_outputs=torch.ones_like(T_bc2), 
                                     create_graph=True, retain_graph=True)[0]
    dTdx_bc2_norm = grad_T_bc2[:, 0:1] / L_char
    dTdy_bc2_norm = grad_T_bc2[:, 1:2] / L_char
    
    x_real = x_bc2[:, 0:1] * L_char
    y_real = x_bc2[:, 1:2] * L_char
    r_norm = torch.sqrt(x_real**2 + y_real**2 + 1e-10)
    dTdn = (x_real/r_norm) * dTdx_bc2_norm + (y_real/r_norm) * dTdy_bc2_norm
    L_BC2 = torch.mean(dTdn**2)
    
    L_IC_BC = L_IC + L_BC1 + L_BC2
    
    # ==================== 拓扑约束损失 ====================
    # ρx取值约束
    L_rho_bounds = torch.sum((torch.relu(-rho_x_col) ** 2 + 
                              torch.relu(rho_x_col - 1.0) ** 2) * collocation_weights) / torch.sum(collocation_weights)
    
    # 体积比约束（硬惩罚）
    L_rho_vol, rho_x_avg = compute_volume_constraint_loss(model, collocation_points, collocation_weights, 
                                                         hard_penalty=(stage=="pretrain"))
    
    # 拓扑网络初始化损失（仅预训练阶段）
    L_rho_init = torch.tensor(0.0).to(device)
    if stage == "pretrain":
        # 鼓励ρ_x接近目标体积比
        L_rho_init = (rho_x_avg - phi_total) ** 2 * 100.0
    
    # 拓扑正则化
    L_topo_reg = model.compute_topo_regularization(collocation_points[:, 0:2])
    
    L_topo_constraint = L_rho_bounds + L_rho_vol + L_rho_init + L_topo_reg
    
    # ==================== 优化目标损失 ====================
    L_objective = torch.tensor(0.0).to(device)
    if compute_objective:
        x_obj, weight_obj = sample_collocation_points(N_obj)
        x_obj = x_obj.requires_grad_(True)
        ux_obj, uy_obj, p_obj, T_obj, rho_x_obj = model(x_obj)
        T_obj = torch.clamp(T_obj, 285.0, 365.0)
        weight_obj = weight_obj.unsqueeze(1)
        
        # 平均温度
        T_ave = torch.sum(T_obj * weight_obj) / torch.sum(weight_obj)
        L_obj1 = T_ave ** 2
        
        # 温度均方差
        T_var = torch.sum(((T_obj - T_ave) ** 2) * weight_obj) / torch.sum(weight_obj)
        L_obj2 = T_var
        
        # 火积耗散（修正为论文形式）
        grad_T_obj = torch.autograd.grad(T_obj, x_obj, grad_outputs=torch.ones_like(T_obj),
                                         create_graph=True, retain_graph=True)[0]
        dTdx_obj_norm = grad_T_obj[:, 0:1] / L_char
        dTdy_obj_norm = grad_T_obj[:, 1:2] / L_char
        
        d2Tdx2_obj_norm = torch.autograd.grad(dTdx_obj_norm, x_obj, grad_outputs=torch.ones_like(dTdx_obj_norm),
                                              create_graph=True, retain_graph=True)[0][:, 0:1] / L_char
        d2Tdy2_obj_norm = torch.autograd.grad(dTdy_obj_norm, x_obj, grad_outputs=torch.ones_like(dTdy_obj_norm),
                                              create_graph=True, retain_graph=True)[0][:, 1:2] / L_char
        
        laplacian_T = d2Tdx2_obj_norm + d2Tdy2_obj_norm
        _, lambda_total_obj, _, _, _, _, _ = compute_thermo_props(T_obj, rho_x_obj)
        
        # φ_g = ∫λ∇²T dA（按论文式(16)）
        phi_g = torch.sum(lambda_total_obj * laplacian_T * weight_obj) / torch.sum(weight_obj)
        L_obj3 = w1 * L_obj1 + w2 * L_obj2 + w3 * phi_g ** 2
        
        if case == 1:
            L_objective = L_obj1
        elif case == 2:
            L_objective = L_obj2
        else:
            L_objective = L_obj3
    
    # ==================== 自适应加权 ====================
    # 根据损失量级动态调整权重
    adaptive_weight_pde = 1.0 / (torch.log(L_PDE.detach() + 10.0) + 1.0)
    adaptive_weight_bc = 10.0 / (torch.log(L_IC_BC.detach() + 10.0) + 1.0)
    adaptive_weight_topo = 100.0 / (torch.log(L_topo_constraint.detach() + 1.0) + 1.0)
    
    # 总损失
    if compute_objective:
        L_total = (adaptive_weight_pde * L_PDE + 
                  adaptive_weight_bc * L_IC_BC + 
                  adaptive_weight_topo * L_topo_constraint + 
                  lambda4 * L_objective)
    else:
        L_total = (adaptive_weight_pde * L_PDE + 
                  adaptive_weight_bc * L_IC_BC + 
                  adaptive_weight_topo * L_topo_constraint)
    
    return L_total, L_PDE, L_IC_BC, L_topo_constraint, L_objective, rho_x_avg

In [13]:
def train_model(model, epochs_pretrain=5000, epochs_topo=2000, epochs_joint=3000, epochs_fine=1000):
    """四阶段训练策略"""
    
    # ==================== 阶段1：固定拓扑，训练PDE+IC/BC ====================
    print("="*50)
    print("阶段1：固定拓扑（均匀分布），训练PDE+IC/BC")
    print("="*50)
    
    # 固定拓扑网络参数
    for param in model.topo_net.parameters():
        param.requires_grad = False
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=50)
    
    collocation_points, collocation_weights = sample_collocation_points(N_mass)
    ic_points, ic_weights = sample_initial_condition(N_IC)
    bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
    
    for epoch in range(epochs_pretrain):
        if epoch % 200 == 0 and epoch != 0:
            collocation_points, collocation_weights = sample_collocation_points(N_mass)
            ic_points, ic_weights = sample_initial_condition(N_IC)
            bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
        
        L_total, L_PDE, L_IC_BC, L_topo, _, rho_x_avg = compute_loss(
            model, collocation_points, collocation_weights,
            ic_points, ic_weights, bc1_points, bc2_points,
            heat_storage=True, compute_objective=False, stage="pretrain"
        )
        
        optimizer.zero_grad()
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step(L_total)
        
        if (epoch + 1) % 100 == 0:
            print(f"阶段1 Epoch [{epoch+1}/{epochs_pretrain}] | "
                  f"总损失: {L_total.item():.2f} | "
                  f"PDE损失: {L_PDE.item():.2f} | "
                  f"IC/BC损失: {L_IC_BC.item():.2f} | "
                  f"体积比: {rho_x_avg.item():.3f}")
    
    # ==================== 阶段2：固定主网络，训练拓扑网络 ====================
    print("\n" + "="*50)
    print("阶段2：固定主网络，训练拓扑网络")
    print("="*50)
    
    # 解冻拓扑网络，冻结主网络
    for param in model.topo_net.parameters():
        param.requires_grad = True
    for param in model.main_net.parameters():
        param.requires_grad = False
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=50)
    
    for epoch in range(epochs_topo):
        if epoch % 200 == 0:
            collocation_points, collocation_weights = sample_collocation_points(N_mass)
            ic_points, ic_weights = sample_initial_condition(N_IC)
            bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
        
        # 此阶段只关心拓扑约束损失
        _, _, _, L_topo, _, rho_x_avg = compute_loss(
            model, collocation_points, collocation_weights,
            ic_points, ic_weights, bc1_points, bc2_points,
            heat_storage=True, compute_objective=False, stage="topo"
        )
        
        # 添加拓扑网络的直接损失
        L_total = L_topo * 10.0
        
        optimizer.zero_grad()
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step(L_total)
        
        if (epoch + 1) % 100 == 0:
            print(f"阶段2 Epoch [{epoch+1}/{epochs_topo}] | "
                  f"拓扑损失: {L_topo.item():.4f} | "
                  f"体积比: {rho_x_avg.item():.3f}")
    
    # ==================== 阶段3：联合训练 ====================
    print("\n" + "="*50)
    print("阶段3：联合训练")
    print("="*50)
    
    # 解冻所有参数
    for param in model.parameters():
        param.requires_grad = True
    
    optimizer = optim.Adam(model.parameters(), lr=5e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=50)
    
    for epoch in range(epochs_joint):
        if epoch % 200 == 0:
            collocation_points, collocation_weights = sample_collocation_points(N_mass)
            ic_points, ic_weights = sample_initial_condition(N_IC)
            bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
        
        L_total, L_PDE, L_IC_BC, L_topo, _, rho_x_avg = compute_loss(
            model, collocation_points, collocation_weights,
            ic_points, ic_weights, bc1_points, bc2_points,
            heat_storage=True, compute_objective=False, stage="joint"
        )
        
        optimizer.zero_grad()
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step(L_total)
        
        if (epoch + 1) % 100 == 0:
            print(f"阶段3 Epoch [{epoch+1}/{epochs_joint}] | "
                  f"总损失: {L_total.item():.4f} | "
                  f"PDE损失: {L_PDE.item():.4f} | "
                  f"拓扑损失: {L_topo.item():.4f} | "
                  f"体积比: {rho_x_avg.item():.3f}")
    
    # ==================== 阶段4：引入优化目标 ====================
    print("\n" + "="*50)
    print("阶段4：优化目标训练")
    print("="*50)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    for epoch in range(epochs_fine):
        if epoch % 100 == 0:
            collocation_points, collocation_weights = sample_collocation_points(N_mass)
            ic_points, ic_weights = sample_initial_condition(N_IC)
            bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
        
        L_total, L_PDE, L_IC_BC, L_topo, L_obj, rho_x_avg = compute_loss(
            model, collocation_points, collocation_weights,
            ic_points, ic_weights, bc1_points, bc2_points,
            heat_storage=True, compute_objective=True, stage="fine"
        )
        
        optimizer.zero_grad()
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        
        if (epoch + 1) % 50 == 0:
            print(f"阶段4 Epoch [{epoch+1}/{epochs_fine}] | "
                  f"总损失: {L_total.item():.4f} | "
                  f"PDE损失: {L_PDE.item():.4f} | "
                  f"目标损失: {L_obj.item():.4f} | "
                  f"体积比: {rho_x_avg.item():.3f}")
    
    # 保存模型
    torch.save(model.state_dict(), f"topo_pinn_case{case}_trained.pth")
    print(f"\n训练完成！模型已保存为 topo_pinn_case{case}_trained.pth")

In [14]:
def visualize_results(model, save_path="./results"):
    """可视化结果"""
    import os
    os.makedirs(save_path, exist_ok=True)
    
    # 生成网格
    r = np.linspace(r0, r1, 100)
    theta = np.linspace(0, 2*np.pi, 100)
    R, Theta = np.meshgrid(r, theta)
    X = R * np.cos(Theta)
    Y = R * np.sin(Theta)
    
    # 关键时间点
    tau_list = [100, 300, 500, 751, 1000]
    
    for tau in tau_list:
        tau_star = tau / t_char
        x_input = np.hstack([
            X.reshape(-1, 1) / L_char,
            Y.reshape(-1, 1) / L_char,
            np.full((10000, 1), tau_star)
        ])
        x_tensor = torch.tensor(x_input, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            ux, uy, p, T, rho_x = model(x_tensor)
            phi = model.compute_phi_from_T(T)
        
        T_grid = T.detach().cpu().numpy().reshape(100, 100)
        phi_grid = phi.detach().cpu().numpy().reshape(100, 100)
        rho_x_grid = rho_x.detach().cpu().numpy().reshape(100, 100)
        
        # 绘图
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 3, 1)
        contourf = plt.contourf(X, Y, T_grid, cmap='jet', vmin=290, vmax=360)
        plt.colorbar(contourf, label='Temperature (K)')
        plt.title(f'Temperature (τ={tau}s)')
        plt.axis('equal')
        
        plt.subplot(1, 3, 2)
        contourf = plt.contourf(X, Y, phi_grid, cmap='viridis', vmin=0, vmax=1)
        plt.colorbar(contourf, label='Liquid Fraction φ')
        plt.title(f'Liquid Fraction (τ={tau}s)')
        plt.axis('equal')
        
        plt.subplot(1, 3, 3)
        contourf = plt.contourf(X, Y, rho_x_grid, cmap='binary', vmin=0, vmax=1)
        plt.colorbar(contourf, label='Topology Density ρ_x')
        plt.title(f'Topology (τ={tau}s)')
        plt.axis('equal')
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f'result_tau_{tau}s.png'), dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"可视化结果已保存到 {save_path}")

In [15]:
if __name__ == "__main__":
    # 初始化模型
    model = TopoPINN(hidden_layers=6, hidden_dim=256).to(device)
    
    # 四阶段训练
    train_model(
        model, 
        epochs_pretrain=3000,    # 阶段1：预训练
        epochs_topo=2000,        # 阶段2：拓扑训练
        epochs_joint=2000,       # 阶段3：联合训练
        epochs_fine=1000         # 阶段4：精细优化
    )
    
    # 加载模型并可视化
    model.load_state_dict(torch.load(f"topo_pinn_case{case}_trained.pth"))
    model.eval()
    visualize_results(model, save_path="./topo_pinn_results")
    
    print("\n训练+可视化完成！")

阶段1：固定拓扑（均匀分布），训练PDE+IC/BC
阶段1 Epoch [100/3000] | 总损失: 817262.06 | PDE损失: 14119993.00 | IC/BC损失: 7802.22 | 体积比: 0.494
阶段1 Epoch [200/3000] | 总损失: 294594.78 | PDE损失: 4686781.00 | IC/BC损失: 7163.19 | 体积比: 0.494
阶段1 Epoch [300/3000] | 总损失: 145024.70 | PDE损失: 1185001.50 | IC/BC损失: 79866.26 | 体积比: 0.505
阶段1 Epoch [400/3000] | 总损失: 16053.91 | PDE损失: 46515.67 | IC/BC损失: 11536.08 | 体积比: 0.505
阶段1 Epoch [500/3000] | 总损失: 15086.08 | PDE损失: 73677.82 | IC/BC损失: 8153.88 | 体积比: 0.498
阶段1 Epoch [600/3000] | 总损失: 9226.59 | PDE损失: 1671.94 | IC/BC损失: 8128.91 | 体积比: 0.498
阶段1 Epoch [700/3000] | 总损失: 9186.76 | PDE损失: 738.74 | IC/BC损失: 8125.47 | 体积比: 0.507
阶段1 Epoch [800/3000] | 总损失: 9143.76 | PDE损失: 404.92 | IC/BC损失: 8121.43 | 体积比: 0.507
阶段1 Epoch [900/3000] | 总损失: 9063.50 | PDE损失: 390.63 | IC/BC损失: 8119.55 | 体积比: 0.497
阶段1 Epoch [1000/3000] | 总损失: 9057.95 | PDE损失: 360.99 | IC/BC损失: 8117.46 | 体积比: 0.497
阶段1 Epoch [1100/3000] | 总损失: 9054.54 | PDE损失: 251.64 | IC/BC损失: 8115.45 | 体积比: 0.498
阶段1 Epoch [1200/300